# 02 读取 DPM 数据并进行 Kriging 插值恢复（批量处理版）

本代码直接读取 RadioMapSeer 原始数据中的 DPM 路径损耗图，并提供可直接调用的 Kriging 插值函数。

实验流程：

1. 从 test.csv 读取所有测试样本列表
2. 对每个样本，按设定的采样比例进行 Kriging 插值恢复
3. 保存每个样本的恢复结果图片和误差指标到 CSV 文件


In [ ]:
from pathlib import Path
import json
import time
import csv
from collections import OrderedDict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd

# =========================
# 1. 基础配置：按自己的数据路径修改
# =========================

# RadioMapSeer 根目录。目录下应包含 gain、png、antenna 等文件夹。
DATA_ROOT = Path(r"G:\SvT\RadioMapSeer")

# test.csv 路径
TEST_CSV_PATH = Path(r"test.csv")

# 输出目录
OUTPUT_DIR = Path(r"kriging_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# 本实验默认使用 DPM 数据。也可改为 IRT2 或 IRT4。
MODE = "DPM"

# 要测试的采样比例列表
SAMPLE_RATIOS = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]

# 随机种子。修改后采样点位置会变化。
SEED = 20260907

# 是否只在非建筑区域采样和评价。建议保持 True。
EXCLUDE_BUILDINGS = True
BUILDING_THRESHOLD = 0.5

# Kriging 参数。NEIGHBORS 越大越慢，恢复可能更平滑。
NEIGHBORS = 16
RANGE_PARAM = 30.0
NUGGET = 1e-5
COVARIANCE = "exponential"  # 可选: "exponential" 或 "gaussian"
CHUNK_SIZE = 1024
CACHE_SIZE = 20000

# RadioMapSeer PNG 灰度值到路径损耗 dB 的转换参数。
PATHLOSS_MAX_DB = -47.84
PATHLOSS_TRUNC_DB = -147.0
IMAGE_SIZE_M = 256

# 中文字体设置，避免 matplotlib 中文乱码。
candidate_fonts = ["Microsoft YaHei", "SimHei", "SimSun", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans"]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
chosen_font = next((font for font in candidate_fonts if font in available_fonts), "DejaVu Sans")
plt.rcParams["font.sans-serif"] = [chosen_font] + candidate_fonts
plt.rcParams["axes.unicode_minus"] = False

print("数据目录:", DATA_ROOT)
print("输出目录:", OUTPUT_DIR)
print("Matplotlib 中文字体:", chosen_font)


In [ ]:
# =========================
# 2. RadioMapSeer 路径与数据读取
# =========================

MODES = {
    "DPM": {"folder": DATA_ROOT / "gain" / "DPM", "num_maps": 701, "num_tx": 80},
    "IRT2": {"folder": DATA_ROOT / "gain" / "IRT2", "num_maps": 701, "num_tx": 80},
    "IRT4": {"folder": DATA_ROOT / "gain" / "IRT4", "num_maps": 701, "num_tx": 2},
}


def gray_to_pathloss_db(gray: np.ndarray) -> np.ndarray:
    gray_float = gray.astype(np.float32)
    if gray_float.max(initial=0) > 1.0:
        gray_float = gray_float / 255.0
    return gray_float * (PATHLOSS_MAX_DB - PATHLOSS_TRUNC_DB) + PATHLOSS_TRUNC_DB


def xy_to_row_col(x: float, y: float, image_size: int = IMAGE_SIZE_M) -> tuple[int, int]:
    # RadioMapSeer 的坐标原点在左下角，numpy 图像数组原点在左上角。
    row = image_size - 1 - int(y)
    col = int(x)
    return row, col


def read_gray(path: str | Path, normalize: bool = True) -> np.ndarray:
    with Image.open(path) as image:
        array = np.asarray(image.convert("L"))
    if normalize:
        return array.astype(np.float32) / 255.0
    return array


def load_raw_dpm_sample(map_id: int, tx_id: int, mode: str = MODE) -> dict:
    building_path = DATA_ROOT / "png" / "buildings_complete" / f"{map_id}.png"
    antenna_path = DATA_ROOT / "antenna" / f"{map_id}.json"
    ckm_path = MODES[mode]["folder"] / f"{map_id}_{tx_id}.png"

    assert building_path.exists(), f"找不到建筑物地图: {building_path}"
    assert antenna_path.exists(), f"找不到 TX 坐标文件: {antenna_path}"
    assert ckm_path.exists(), f"找不到 CKM 文件: {ckm_path}"

    buildings = read_gray(building_path, normalize=True)
    ckm_gray = read_gray(ckm_path, normalize=False)
    ckm_norm = ckm_gray.astype(np.float32) / 255.0
    ckm_db = gray_to_pathloss_db(ckm_gray)

    all_tx_xy = json.loads(antenna_path.read_text(encoding="utf-8"))
    tx_x, tx_y = all_tx_xy[tx_id]
    tx_row, tx_col = xy_to_row_col(tx_x, tx_y)

    tx_mask = np.zeros_like(ckm_norm, dtype=np.float32)
    if 0 <= tx_row < tx_mask.shape[0] and 0 <= tx_col < tx_mask.shape[1]:
        tx_mask[tx_row, tx_col] = 1.0

    return {
        "map_id": map_id,
        "tx_id": tx_id,
        "mode": mode,
        "building_path": building_path,
        "antenna_path": antenna_path,
        "ckm_path": ckm_path,
        "buildings": buildings,
        "ckm_norm": ckm_norm,
        "ckm_db": ckm_db,
        "tx_xy": (tx_x, tx_y),
        "tx_row_col": (tx_row, tx_col),
        "tx_mask": tx_mask,
    }

In [ ]:
# =========================
# 3. 采样、指标与局部 Ordinary Kriging
# =========================

def valid_region_mask(buildings: np.ndarray, shape: tuple[int, int]) -> np.ndarray:
    valid = np.ones(shape, dtype=bool)
    if EXCLUDE_BUILDINGS:
        valid &= buildings <= BUILDING_THRESHOLD
    return valid


def covariance(distance: np.ndarray, range_param: float, model: str) -> np.ndarray:
    distance = np.asarray(distance, dtype=np.float64)
    if model == "gaussian":
        return np.exp(-((distance / range_param) ** 2))
    return np.exp(-(distance / range_param))


def nearest_chunks(query_coords: np.ndarray, sample_coords: np.ndarray, k: int, chunk_size: int):
    k = min(k, len(sample_coords))
    sample_coords64 = sample_coords.astype(np.float64)
    for start in range(0, len(query_coords), chunk_size):
        stop = min(start + chunk_size, len(query_coords))
        query = query_coords[start:stop].astype(np.float64)
        diff = query[:, None, :] - sample_coords64[None, :, :]
        dist2 = np.sum(diff * diff, axis=2)
        idx = np.argpartition(dist2, kth=k - 1, axis=1)[:, :k]
        selected_dist2 = np.take_along_axis(dist2, idx, axis=1)
        order = np.argsort(selected_dist2, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        selected_dist2 = np.take_along_axis(selected_dist2, order, axis=1)
        yield start, idx, np.sqrt(selected_dist2)


def lru_get(cache: OrderedDict, key: tuple[int, ...]):
    value = cache.get(key)
    if value is not None:
        cache.move_to_end(key)
    return value


def lru_put(cache: OrderedDict, key: tuple[int, ...], value: np.ndarray, max_size: int) -> None:
    if max_size <= 0:
        return
    cache[key] = value
    cache.move_to_end(key)
    while len(cache) > max_size:
        cache.popitem(last=False)


def kriging_inverse_for_neighbors(neighbor_coords: np.ndarray) -> np.ndarray:
    k = len(neighbor_coords)
    diff = neighbor_coords[:, None, :] - neighbor_coords[None, :, :]
    distances = np.sqrt(np.sum(diff * diff, axis=2))
    cov = covariance(distances, range_param=RANGE_PARAM, model=COVARIANCE)
    cov.flat[:: k + 1] += NUGGET

    system = np.ones((k + 1, k + 1), dtype=np.float64)
    system[:k, :k] = cov
    system[k, k] = 0.0
    try:
        return np.linalg.inv(system)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(system)


def local_ordinary_kriging(sample_coords: np.ndarray, sample_values: np.ndarray, query_coords: np.ndarray) -> np.ndarray:
    if len(sample_coords) == 0:
        raise ValueError("至少需要一个采样点。")

    predictions = np.empty(len(query_coords), dtype=np.float32)
    sample_values64 = sample_values.astype(np.float64)
    inverse_cache = OrderedDict()

    for chunk_start, neighbor_idx, neighbor_dist in nearest_chunks(
        query_coords=query_coords,
        sample_coords=sample_coords,
        k=NEIGHBORS,
        chunk_size=CHUNK_SIZE,
    ):
        for local_row in range(neighbor_idx.shape[0]):
            out_idx = chunk_start + local_row
            idx = neighbor_idx[local_row]
            dists = neighbor_dist[local_row]

            if dists[0] < 1e-12:
                predictions[out_idx] = sample_values64[idx[0]]
                continue

            key = tuple(int(i) for i in idx)
            inv_system = lru_get(inverse_cache, key)
            if inv_system is None:
                inv_system = kriging_inverse_for_neighbors(sample_coords[idx])
                lru_put(inverse_cache, key, inv_system, CACHE_SIZE)

            rhs = np.ones(len(idx) + 1, dtype=np.float64)
            rhs[:-1] = covariance(dists, range_param=RANGE_PARAM, model=COVARIANCE)
            weights = inv_system @ rhs
            predictions[out_idx] = np.dot(weights[:-1], sample_values64[idx])

    return predictions


def random_sample_flat(valid_mask: np.ndarray, ratio: float, rng: np.random.Generator) -> np.ndarray:
    if not 0 < ratio <= 1:
        raise ValueError("采样比例 ratio 应满足 0 < ratio <= 1。")
    valid_flat = np.flatnonzero(valid_mask.ravel())
    count = max(1, int(np.ceil(len(valid_flat) * ratio)))
    return rng.choice(valid_flat, size=count, replace=False)


def sample_arrays_from_flat(target: np.ndarray, sample_flat: np.ndarray):
    rows, cols = np.unravel_index(sample_flat, target.shape)
    sample_coords = np.column_stack([rows, cols]).astype(np.float32)
    sample_values = target[rows, cols].astype(np.float32)
    sample_mask = np.zeros(target.shape, dtype=bool)
    sample_mask[rows, cols] = True
    return sample_coords, sample_values, sample_mask


def metric_report(pred_norm: np.ndarray, target_norm: np.ndarray, mask: np.ndarray) -> dict[str, float]:
    pred_norm = np.clip(pred_norm, 0.0, 1.0)
    target_norm = target_norm.astype(np.float32)
    diff_norm = pred_norm[mask].astype(np.float64) - target_norm[mask].astype(np.float64)
    nmse_norm = float(np.sum(diff_norm ** 2) / (np.sum(target_norm[mask].astype(np.float64) ** 2) + 1e-12))

    pred_db = gray_to_pathloss_db(pred_norm)
    target_db = gray_to_pathloss_db(target_norm)
    diff_db = pred_db[mask].astype(np.float64) - target_db[mask].astype(np.float64)
    rmse_db = float(np.sqrt(np.mean(diff_db ** 2)))
    mae_db = float(np.mean(np.abs(diff_db)))
    return {
        "nmse_norm": nmse_norm,
        "rmse_db": rmse_db,
        "mae_db": mae_db,
    }


def run_kriging_reconstruction(
    target_norm: np.ndarray,
    buildings: np.ndarray,
    sample_ratio: float,
    seed: int = 2026,
) -> dict:
    """按指定比例随机采样，并调用 Kriging 函数恢复一张完整 CKM。"""
    valid_mask = valid_region_mask(buildings, target_norm.shape)
    rng = np.random.default_rng(seed)
    sample_flat = random_sample_flat(valid_mask, sample_ratio, rng)
    sample_coords, sample_values, sample_mask = sample_arrays_from_flat(target_norm, sample_flat)
    query_coords = np.column_stack(np.nonzero(valid_mask)).astype(np.float32)

    started = time.perf_counter()
    valid_pred = local_ordinary_kriging(sample_coords, sample_values, query_coords)
    elapsed = time.perf_counter() - started

    pred_norm = np.full(target_norm.shape, float(np.nanmin(target_norm)), dtype=np.float32)
    pred_norm[valid_mask] = np.clip(valid_pred, 0.0, 1.0)
    metrics = metric_report(pred_norm, target_norm, valid_mask)

    return {
        "ratio": sample_ratio,
        "sample_count": int(sample_mask.sum()),
        "prediction_norm": pred_norm,
        "prediction_db": gray_to_pathloss_db(pred_norm),
        "sample_mask": sample_mask,
        "valid_mask": valid_mask,
        "elapsed_sec": elapsed,
        "metrics": metrics,
    }

In [ ]:
# =========================
# 4. 批量处理函数
# =========================

def load_test_samples(csv_path: Path) -> list[dict]:
    """从 test.csv 加载所有测试样本信息"""
    samples = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            samples.append({
                "sample_id": row["sample_id"],
                "scene_id": int(row["scene_id"]),
                "tx_id": int(row["tx_id"]),
            })
    return samples


def save_kriging_result(
    sample_id: str,
    map_id: int,
    tx_id: int,
    ratio: float,
    target_db: np.ndarray,
    result: dict,
    output_dir: Path
) -> None:
    """保存 Kriging 恢复结果图片"""
    valid_mask = result["valid_mask"]
    
    # 创建样本子目录
    sample_dir = output_dir / sample_id
    sample_dir.mkdir(exist_ok=True)
    
    # 准备数据
    target_db_show = np.where(valid_mask, target_db, float(np.nanmin(target_db)))
    vmin = float(np.nanmin(target_db))
    vmax = float(np.nanmax(target_db))
    error = np.where(valid_mask, np.abs(result["prediction_db"] - target_db), 0.0)
    error_max = float(np.max(error))
    error_max = max(error_max, 1e-12)
    
    metrics = result["metrics"]
    
    # 创建图形
    fig, axes = plt.subplots(
        2, 3,
        figsize=(8.8, 8.0),
        gridspec_kw={"width_ratios": [1, 1, 0.055]},
        constrained_layout=True,
    )
    
    ckm_axes = axes[0, :2]
    err_axes = axes[1, :2]
    cax_ckm = axes[0, -1]
    cax_err = axes[1, -1]
    
    # 原始 CKM
    im_ckm = ckm_axes[0].imshow(target_db_show, cmap="viridis", vmin=vmin, vmax=vmax)
    ckm_axes[0].set_title("原始 CKM", fontsize=13)
    ckm_axes[0].axis("off")
    
    # 零误差参考
    zero_error = np.zeros_like(target_db, dtype=np.float32)
    err_axes[0].imshow(zero_error, cmap="magma", vmin=0, vmax=error_max)
    err_axes[0].set_title("绝对误差参考: 0 dB", fontsize=13)
    err_axes[0].axis("off")
    
    # Kriging 恢复结果
    pred_db_show = np.where(valid_mask, result["prediction_db"], vmin)
    ckm_axes[1].imshow(pred_db_show, cmap="viridis", vmin=vmin, vmax=vmax)
    ckm_axes[1].set_title(
        f"Kriging {ratio*100:g}%\nNMSE={metrics['nmse_norm']:.4g}",
        fontsize=13,
    )
    ckm_axes[1].axis("off")
    
    # 误差 + 采样点
    err_axes[1].imshow(error, cmap="magma", vmin=0, vmax=error_max)
    sampled_rows, sampled_cols = np.nonzero(result["sample_mask"] & valid_mask)
    err_axes[1].scatter(sampled_cols, sampled_rows, s=3, c="cyan", alpha=0.65)
    err_axes[1].set_title(
        f"误差 + 采样点\nRMSE={metrics['rmse_db']:.3g} dB",
        fontsize=13,
    )
    err_axes[1].axis("off")
    
    # Colorbar
    cb1 = fig.colorbar(im_ckm, cax=cax_ckm)
    cb1.set_label("路径损耗 (dB)", fontsize=12)
    cb1.ax.tick_params(labelsize=11)
    
    im_err = err_axes[1].images[0]
    cb2 = fig.colorbar(im_err, cax=cax_err)
    cb2.set_label("绝对误差 (dB)", fontsize=12)
    cb2.ax.tick_params(labelsize=11)
    
    # 保存图片
    ratio_str = f"{ratio:.4f}".replace('.', '_')
    fig_path = sample_dir / f"map{map_id}_tx{tx_id}_ratio{ratio_str}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return fig_path

In [ ]:
# =========================
# 5. 执行批量处理
# =========================

# 加载测试样本列表
test_samples = load_test_samples(TEST_CSV_PATH)
print(f"共加载 {len(test_samples)} 个测试样本")
print(f"将要测试 {len(SAMPLE_RATIOS)} 种采样比例: {[f'{r*100:.2f}%' for r in SAMPLE_RATIOS]}")

# 存储所有结果
all_results = []

# 开始批量处理
for idx, sample_info in enumerate(test_samples):
    map_id = sample_info["scene_id"]
    tx_id = sample_info["tx_id"]
    sample_id = sample_info["sample_id"]
    
    print(f"\n[{idx+1}/{len(test_samples)}] 处理样本: {sample_id} (map={map_id}, tx={tx_id})")
    
    # 加载数据
    try:
        data = load_raw_dpm_sample(map_id, tx_id, MODE)
        target_norm = data["ckm_norm"]
        target_db = data["ckm_db"]
        buildings = data["buildings"]
    except Exception as e:
        print(f"  ❌ 加载数据失败: {e}")
        continue
    
    # 对每种采样比例进行处理
    for ratio in SAMPLE_RATIOS:
        print(f"  📊 采样比例: {ratio*100:.2f}%", end="", flush=True)
        
        try:
            # 执行 Kriging 重建
            result = run_kriging_reconstruction(
                target_norm=target_norm,
                buildings=buildings,
                sample_ratio=ratio,
                seed=SEED,
            )
            
            m = result["metrics"]
            print(f" -> RMSE={m['rmse_db']:.3f} dB, MAE={m['mae_db']:.3f} dB, time={result['elapsed_sec']:.2f}s")
            
            # 保存结果图片
            fig_path = save_kriging_result(
                sample_id=sample_id,
                map_id=map_id,
                tx_id=tx_id,
                ratio=ratio,
                target_db=target_db,
                result=result,
                output_dir=OUTPUT_DIR,
            )
            
            # 记录结果
            all_results.append({
                "sample_id": sample_id,
                "map_id": map_id,
                "tx_id": tx_id,
                "ratio": ratio,
                "sample_count": result["sample_count"],
                "nmse": m["nmse_norm"],
                "rmse_db": m["rmse_db"],
                "mae_db": m["mae_db"],
                "time_sec": result["elapsed_sec"],
                "fig_path": str(fig_path),
            })
            
        except Exception as e:
            print(f" -> ❌ 失败: {e}")
            import traceback
            traceback.print_exc()

# =========================
# 6. 保存结果汇总
# =========================

# 保存详细结果到 CSV
df = pd.DataFrame(all_results)
csv_path = OUTPUT_DIR / "kriging_batch_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n📁 详细结果已保存到: {csv_path}")

# 打印汇总统计
print("\n" + "="*60)
print("📊 汇总统计 (按采样比例分组)")
print("="*60)

summary = df.groupby("ratio").agg({
    "rmse_db": ["mean", "std"],
    "mae_db": ["mean", "std"],
    "nmse": ["mean", "std"],
    "time_sec": ["mean", "std"],
    "sample_count": ["mean", "std"],
}).round(4)

# 美化打印
for ratio in sorted(df["ratio"].unique()):
    subset = df[df["ratio"] == ratio]
    print(f"\n  采样比例: {ratio*100:.2f}% ({len(subset)} 个样本)")
    print(f"    RMSE: {subset['rmse_db'].mean():.4f} ± {subset['rmse_db'].std():.4f} dB")
    print(f"    MAE : {subset['mae_db'].mean():.4f} ± {subset['mae_db'].std():.4f} dB")
    print(f"    NMSE: {subset['nmse'].mean():.6f} ± {subset['nmse'].std():.6f}")
    print(f"    时间: {subset['time_sec'].mean():.2f} ± {subset['time_sec'].std():.2f} s")

# 保存汇总统计
summary_path = OUTPUT_DIR / "kriging_summary.csv"
summary.to_csv(summary_path)
print(f"\n📁 汇总统计已保存到: {summary_path}")

print("\n✅ 批量处理完成！")